# Notebook 05: Pipeline Performance & Latency Measurement

**Persona:** ML Engineer / Data Engineer

**Purpose:** Demonstrate and measure the full incremental Feature Store pipeline:
1. Deploy an incremental data generator with live-adjustable batch sizes
2. Enable ROW_TIMESTAMP across the pipeline for precise latency measurement
3. Measure end-to-end latency: source tables → DT Feature Views → Online Feature Tables
4. Adjust data ingestion scale live and observe pipeline behaviour
5. Visualise pipeline latency at every stage

**Prerequisites:** Run Notebooks 00, 01, 02, and 03 first.

**Key concept: ROW_TIMESTAMP** — Snowflake's `METADATA$ROW_LAST_COMMIT_TIME` records the exact timestamp when each row was committed. By enabling this on both source tables and Dynamic Tables, we can measure pipeline propagation delay at every stage.

## 1. Setup

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    sys.path.insert(0, ".")
    from feature_definitions.config import get_session, ROLES
    session = get_session(role=ROLES["admin"])

sys.path.insert(0, ".") if "." not in sys.path else None

from feature_definitions.config import get_config, ROLES
import pandas as pd

cfg = get_config("DEV")
session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

print(f"Role: {session.get_current_role()}")
print(f"Database: {cfg['database']}")
print(f"Source schema: {cfg['source_schema']}")
print(f"Feature Store schema: {cfg['fs_schema']}")
print(f"Admin schema: {cfg['admin_schema']}")

## 2. Deploy the Incremental Data Generator

The generator infrastructure consists of three admin tables:
- **GENERATION_CONFIG**: Live-adjustable batch sizes (sessions, events, orders per batch)
- **GENERATION_STATE**: Tracks ID counters and running totals across batches
- **GENERATION_LOG**: Records each batch's results (row counts, duration, status)

The stored procedure reads from `GENERATION_CONFIG` at the start of each cycle, so you can adjust scale while the task is running.

In [ ]:
from feature_definitions.generator import (
    create_admin_tables,
    seed_state_from_existing_data,
    deploy_stored_procedure,
    deploy_task,
    set_scale,
    generate_batch,
)

# Create admin tables
create_admin_tables(session, "DEV")
print("Admin tables created: GENERATION_CONFIG, GENERATION_STATE, GENERATION_LOG")

# Seed state from existing bulk-loaded data so IDs don't collide
counters = seed_state_from_existing_data(session, "DEV")
print(f"\nState seeded from existing data:")
for k, v in counters.items():
    print(f"  {k}: {v}")

# Deploy the stored procedure
deploy_stored_procedure(session, "DEV")
print("\nStored procedure GENERATE_INCREMENTAL_BATCH deployed")

# Deploy the task (starts SUSPENDED — we'll resume it later)
deploy_task(session, "DEV", schedule="1 MINUTE")
print("Task INCREMENTAL_DATA_TASK deployed (SUSPENDED)")

### Check current generator configuration

In [ ]:
db = cfg["database"]
admin = cfg["admin_schema"]

config_df = session.sql(f"""
    SELECT SESSIONS_PER_BATCH, EVENTS_PER_SESSION_MIN, EVENTS_PER_SESSION_MAX,
           ORDERS_PER_BATCH, ITEMS_PER_ORDER_MIN, ITEMS_PER_ORDER_MAX, IS_ENABLED
    FROM {db}.{admin}.GENERATION_CONFIG WHERE ID = 1
""")
config_df.show()

## 3. Enable ROW_TIMESTAMP

Snowflake's ROW_TIMESTAMP feature exposes `METADATA$ROW_LAST_COMMIT_TIME` on each row, giving the exact commit timestamp. We enable this on:
- **Source tables** (SESSIONS, EVENTS, ORDERS, ORDER_ITEMS)
- **Dynamic Table Feature Views** (all 9 DT-backed FVs)

> **Note:** Rows that existed before enablement will have NULL timestamps. Only new/updated rows get commit times.

In [ ]:
from feature_definitions.latency import (
    enable_row_timestamp_on_sources,
    enable_row_timestamp_on_dts,
    get_source_freshness,
    get_dt_freshness,
    measure_stage_latency,
    get_oft_freshness,
    get_batch_latency,
    pipeline_summary,
)

# Enable on source tables (needs table ownership — ACCOUNTADMIN or table owner)
session.sql("USE ROLE ACCOUNTADMIN").collect()
results = enable_row_timestamp_on_sources(session, "DEV")
for r in results:
    print(r)

# Enable on DTs (owned by FS_DEV_ROLE)
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()
results = enable_row_timestamp_on_dts(session, "DEV")
for r in results:
    print(r)

# Switch back to admin
session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

## 4. Run Initial Batches

Let's generate a few batches of data to establish a baseline. The `generate_batch()` function runs locally (same logic as the stored procedure) so we can iterate quickly.

In [ ]:
# Run 3 batches at default scale (50 sessions, 5 orders per batch)
for i in range(3):
    result = generate_batch(session, "DEV")
    print(f"Batch {i+1}: {result['status']} — "
          f"{result.get('sessions', 0)} sessions, "
          f"{result.get('events', 0)} events, "
          f"{result.get('orders', 0)} orders, "
          f"{result.get('duration_ms', 0)}ms")
    if i < 2:
        time.sleep(2)

### Check source table freshness

After inserting batches, `METADATA$ROW_LAST_COMMIT_TIME` on source tables shows when data was last committed.

In [ ]:
freshness = get_source_freshness(session, "DEV")
fresh_df = pd.DataFrame(freshness)
print("Source table freshness:")
print(fresh_df[["TABLE", "LAST_COMMIT_TIME", "AGE_SECONDS"]].to_string(index=False))

## 5. Adjust Scale Live

The generator reads batch sizes from `GENERATION_CONFIG` at the start of each cycle. Changing the config takes effect on the *next* batch — no restart needed.

Let's increase the scale to 200 sessions and 20 orders per batch.

In [ ]:
# Scale up
set_scale(session, "DEV", sessions_per_batch=200, orders_per_batch=20)

# Verify the config change
row = session.sql(f"""
    SELECT SESSIONS_PER_BATCH, ORDERS_PER_BATCH
    FROM {db}.{admin}.GENERATION_CONFIG WHERE ID = 1
""").collect()[0]
print(f"Config updated: sessions_per_batch={row['SESSIONS_PER_BATCH']}, "
      f"orders_per_batch={row['ORDERS_PER_BATCH']}")

# Run 2 batches at the higher scale
for i in range(2):
    result = generate_batch(session, "DEV")
    print(f"Batch (large) {i+1}: {result['status']} — "
          f"{result.get('sessions', 0)} sessions, "
          f"{result.get('events', 0)} events, "
          f"{result.get('orders', 0)} orders")
    if i < 1:
        time.sleep(2)

# Reset to default
set_scale(session, "DEV", sessions_per_batch=50, orders_per_batch=5)
print("\nScale reset to default (50 sessions, 5 orders)")

## 6. Trigger DT Refresh & Measure Latency

Dynamic Tables refresh on their configured `TARGET_LAG`. To see results immediately, we trigger manual refreshes and then measure how long it took for source data to propagate through each stage.

In [ ]:
from feature_definitions.latency import TIER1_DTS, TIER2_DTS

# Trigger refresh on all DTs (needs DT owner role)
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

for dt in TIER1_DTS + TIER2_DTS:
    try:
        session.sql(
            f'ALTER DYNAMIC TABLE {cfg["database"]}.{cfg["fs_schema"]}."{ dt}" REFRESH'
        ).collect()
        print(f"Refresh triggered: {dt}")
    except Exception as e:
        print(f"Refresh failed {dt}: {e}")

session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

print("\nWaiting 60s for refreshes to complete...")
time.sleep(60)

### DT Feature View Freshness

Now that DTs have refreshed with new data, `METADATA$ROW_LAST_COMMIT_TIME` will show when each DT last committed rows.

In [ ]:
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

dt_fresh = get_dt_freshness(session, "DEV")
dt_df = pd.DataFrame(dt_fresh)
print("Dynamic Table Feature View freshness:")
print(dt_df[["TABLE", "LAST_COMMIT_TIME", "AGE_SECONDS"]].to_string(index=False))

session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

### Stage-to-Stage Latency

Compare MAX(METADATA$ROW_LAST_COMMIT_TIME) between source tables and their downstream DT Feature Views to measure propagation delay.

> **Note:** Cross-table ROW_TIMESTAMP comparisons are approximate (Snowflake only guarantees ordering within a single table).

In [ ]:
stages = measure_stage_latency(session, "DEV")
stage_df = pd.DataFrame(stages)
print("Pipeline stage latency:")
print(stage_df[["STAGE", "FROM_TABLE", "TO_TABLE", "FROM_COMMIT", "TO_COMMIT", "LATENCY_SECONDS"]].to_string(index=False))

## 7. Online Feature Table Retrieval Freshness

OFTs provide low-latency serving of the latest feature values. Here we read from the OFTs and check freshness by comparing the backing DT's ROW_TIMESTAMP against the current time.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.feature_store.feature_view import OnlineConfig

fs = FeatureStore(
    session=session,
    database=cfg["database"],
    name=cfg["fs_schema"],
    default_warehouse=cfg["warehouse"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Ensure OFTs are enabled for DT-backed FVs
# OFT on DT-backed FVs — USER_RECENCY_RAW serves raw timestamps for ODT pattern
for fv_name in ["SESSION_BEHAVIOR_FEATURES", "USER_RECENCY_RAW"]:
    try:
        fs.update_feature_view(fv_name, "V01", online_config=OnlineConfig(enable=True))
        print(f"OFT enabled: {fv_name}")
    except Exception as e:
        print(f"OFT for {fv_name}: {e}")

In [ ]:
# Read from Online Feature Tables
print("SESSION_BEHAVIOR_FEATURES (online):")
fv_sbf = fs.get_feature_view("SESSION_BEHAVIOR_FEATURES", "V01")
result = fs.read_feature_view(
    fv_sbf,
    keys=[["sess_00000001"], ["sess_00000010"]],
    store_type="online",
)
result.show()

print("\nUSER_RECENCY_RAW (online — raw timestamps for ODT pattern):")
fv_urf = fs.get_feature_view("USER_RECENCY_RAW", "V01")
result = fs.read_feature_view(
    fv_urf,
    keys=[["usr_00000001"], ["usr_00000010"]],
    store_type="online",
)
result.show()

In [ ]:
# Measure OFT freshness
# - DT last refresh derived from METADATA$ROW_LAST_COMMIT_TIME (ROW_TIMESTAMP)
# - OFT sync: UPDATED_TS comparison for FULL DTs, latest-key check for INCREMENTAL DTs
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

oft_fresh = get_oft_freshness(session, "DEV", fs=fs)
oft_df = pd.DataFrame(oft_fresh)
print("OFT Freshness:")
display_cols = [c for c in ["FEATURE_VIEW", "DT_LAST_COMMIT", "OFT_SYNCED",
                             "DT_MAX_KEY", "OFT_UPDATED_TS",
                             "OFT_DATA_AGE_SECONDS", "DT_TO_OFT_DELTA_SECONDS"] if c in oft_df.columns]
print(oft_df[display_cols].to_string(index=False))

session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

## 8. End-to-End Latency Measurement

Generate one batch, then track data through every pipeline stage to OFT visibility.

Both `SESSION_BEHAVIOR_FEATURES` and `USER_RECENCY_RAW` are DT-backed FVs with OFTs. OFT sync is detected by polling for the latest entity key from the DT to appear in the OFT.

| Stage | Timestamp Source |
|---|---|
| T0 Generate | Wall clock at batch generation |
| T1 Source Commit | `METADATA$ROW_LAST_COMMIT_TIME` on source table |
| T2 DT Commit | `METADATA$ROW_LAST_COMMIT_TIME` on Dynamic Table |
| T3 OFT Visible (recency) | Wall clock when latest `USER_ID` appears in OFT |
| T3 OFT Visible (session) | Wall clock when latest `SESSION_ID` appears in OFT |

In [ ]:
from feature_definitions.latency import measure_end_to_end

print("Running end-to-end latency measurement...")
print("(generates 1 batch, triggers DT refresh, polls OFT — takes 1-3 minutes)\n")

e2e = measure_end_to_end(session, "DEV", fs=fs)

print(f"Batch: {e2e['batch']['status']} — "
      f"{e2e['batch'].get('sessions', 0)} sessions, "
      f"{e2e['batch'].get('orders', 0)} orders")
if e2e.get("sbf_max_key"):
    print(f"  SESSION_BEHAVIOR polling key: {e2e['sbf_max_key']}")
print()

print(f"{'STAGE':<25s} {'TIMESTAMP':<35s} {'LATENCY':>10s}")
print(f"{'-'*25} {'-'*35} {'-'*10}")
print(f"{'T0 Generate':<25s} {str(e2e['T0_generate']):<35s} {'':>10s}")
print(f"{'T1 Source commit':<25s} {str(e2e['T1_source_commit']):<35s} "
      f"{e2e.get('ingest_seconds', 'N/A'):>10s}s" if e2e.get('ingest_seconds') else
      f"{'T1 Source commit':<25s} {str(e2e.get('T1_source_commit')):<35s}")
print(f"{'T2 DT commit':<25s} {str(e2e.get('T2_dt_commit')):<35s} "
      f"{e2e.get('dt_refresh_seconds', 'N/A'):>10s}s" if e2e.get('dt_refresh_seconds') else
      f"{'T2 DT commit':<25s} {str(e2e.get('T2_dt_commit')):<35s}")

if e2e.get("T3_oft_visible_urf"):
    print(f"{'T3 OFT (recency key)':<25s} {str(e2e['T3_oft_visible_urf']):<35s} "
          f"{e2e.get('oft_sync_urf_seconds', 'N/A'):>10}s")
if e2e.get("T3_oft_visible_sbf"):
    print(f"{'T3 OFT (session key)':<25s} {str(e2e['T3_oft_visible_sbf']):<35s} "
          f"{e2e.get('oft_sync_sbf_seconds', 'N/A'):>10}s")

print(f"\n{'='*70}")
if e2e.get("total_e2e_urf_seconds"):
    print(f"Total end-to-end (USER_RECENCY_RAW):  {e2e['total_e2e_urf_seconds']:>6.1f}s")
if e2e.get("total_e2e_sbf_seconds"):
    print(f"Total end-to-end (SESSION_BEHAVIOR):  {e2e['total_e2e_sbf_seconds']:>6.1f}s")

## 9. Batch-Level Latency Analysis

The `GENERATION_LOG` table records every batch the generator produces. By joining with source table ROW_TIMESTAMPs, we can compute per-batch ingestion latency.

In [ ]:
batch_lat = get_batch_latency(session, "DEV", last_n=15)
batch_df = pd.DataFrame(batch_lat)
print("Batch-level performance:")
cols = [c for c in ["LOG_ID", "BATCH_TS", "SESSIONS_GENERATED", "EVENTS_GENERATED",
                     "ORDERS_GENERATED", "DURATION_MS", "INGEST_LATENCY_SECONDS"] if c in batch_df.columns]
print(batch_df[cols].to_string(index=False))

## 10. OFT Serving Benchmark

Measure feature-serving throughput and latency by running concurrent point-lookups against the Online Feature Tables. This emulates real application-server traffic:

- **Dedicated warehouse** (`FS_SERVING_WH`) — XS, multi-cluster auto-scale
- **8 concurrent threads per cluster** — each thread has its own Snowpark session
- Random entity keys are sampled and looked up via `read_feature_view(..., store_type="online")`
- Latency percentiles: **p50, p90, p95, p99** plus overall QPM

The benchmark runs *after* the DT refresh cycle, so OFTs contain fresh data. In production you would run this workload continuously alongside ingestion to validate SLAs.

In [ ]:
from feature_definitions.benchmark import (
    create_serving_warehouse,
    run_benchmark,
    BenchmarkConfig,
)

# Create the serving warehouse (needs elevated role)
session.sql("USE ROLE ACCOUNTADMIN").collect()
create_serving_warehouse(session, "DEV", max_clusters=1)
session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

# Run 60-second benchmark with 8 concurrent threads
bench_cfg = BenchmarkConfig(
    duration_seconds=60,
    threads_per_cluster=8,
    max_clusters=1,
)
bench_result = run_benchmark(session, "DEV", bench_cfg)
bench_result.print_summary()

## 11. Visualise Pipeline Latency

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Stage latency bar chart
stage_data = measure_stage_latency(session, "DEV")
stage_names = [f"{s['FROM_TABLE']}\n→ {s['TO_TABLE']}" for s in stage_data]
latencies = [s.get('LATENCY_SECONDS') or 0 for s in stage_data]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#29B5E8' if l > 0 else '#CCCCCC' for l in latencies]
ax.barh(stage_names, latencies, color=colors)
ax.set_xlabel("Latency (seconds)")
ax.set_title("Source → DT Feature View Propagation Latency")
for i, v in enumerate(latencies):
    ax.text(v + 0.5, i, f"{v}s", va='center', fontsize=9)
plt.tight_layout()
plt.savefig("pipeline_latency_stages.png", dpi=120)
plt.show()
print("Saved: pipeline_latency_stages.png")

In [ ]:
# Batch duration and volume over time
if len(batch_df) > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1.bar(range(len(batch_df)), batch_df["SESSIONS_GENERATED"], color="#29B5E8", alpha=0.7, label="Sessions")
    ax1.bar(range(len(batch_df)), batch_df["ORDERS_GENERATED"], color="#FF6B35", alpha=0.9, label="Orders")
    ax1.set_ylabel("Rows Generated")
    ax1.set_title("Batch Volume Over Time")
    ax1.legend()

    ax2.bar(range(len(batch_df)), batch_df["DURATION_MS"] / 1000, color="#2ECC71", alpha=0.7)
    ax2.set_ylabel("Duration (s)")
    ax2.set_xlabel("Batch #")
    ax2.set_title("Batch Processing Duration")

    plt.tight_layout()
    plt.savefig("pipeline_batch_performance.png", dpi=120)
    plt.show()
    print("Saved: pipeline_batch_performance.png")
else:
    print("No batch data to visualise yet")

## 12. Pipeline Summary

In [ ]:
summary = pipeline_summary(session, "DEV")

print(f"Pipeline Health Summary ({summary['measured_at']})")
print(f"{'='*60}")

print(f"\nSource Tables ({len(summary['source_freshness'])}):")
for s in summary["source_freshness"]:
    age = s.get('AGE_SECONDS')
    status = f"{age}s old" if age else "no data"
    print(f"  {s['TABLE']:20s} {status}")

print(f"\nDT Feature Views ({len(summary['dt_freshness'])}):")
for d in summary["dt_freshness"]:
    age = d.get('AGE_SECONDS')
    status = f"{age}s old" if age else "awaiting refresh"
    print(f"  {d['TABLE']:45s} {status}")

print(f"\nStage Latencies ({len(summary['stage_latency'])}):")
for s in summary["stage_latency"]:
    lat = s.get('LATENCY_SECONDS')
    status = f"{lat}s" if lat is not None else "N/A"
    print(f"  {s['FROM_TABLE']:20s} → {s['TO_TABLE']:40s} {status}")

## 13. Using the Scheduled Task

For continuous operation, the generator can run as a scheduled Snowflake Task. The task calls the stored procedure every minute.

```sql
-- Resume the task to start continuous generation
ALTER TASK FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.INCREMENTAL_DATA_TASK RESUME;

-- Adjust scale while running (takes effect next cycle)
UPDATE FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.GENERATION_CONFIG
SET SESSIONS_PER_BATCH = 500, ORDERS_PER_BATCH = 50
WHERE ID = 1;

-- Suspend when done
ALTER TASK FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.INCREMENTAL_DATA_TASK SUSPEND;
```

The following cell demonstrates starting, running for a few cycles, and stopping the task.

In [ ]:
# Uncomment below to test the scheduled task
# WARNING: This will generate data every minute until suspended

# from feature_definitions.generator import resume_task, suspend_task
# resume_task(session, "DEV")
# print("Task RESUMED — generating data every 1 minute")
# print("Run the following to check progress:")
# print(f"  SELECT * FROM {db}.{admin}.GENERATION_LOG ORDER BY LOG_ID DESC LIMIT 5;")
# print("\nWhen done, run: suspend_task(session, 'DEV')")

## 14. Cleanup Notes

This notebook creates the following objects (all within the existing databases/schemas):
- `CLICKSTREAM_ADMIN.GENERATION_CONFIG` — batch size configuration
- `CLICKSTREAM_ADMIN.GENERATION_STATE` — generator state tracking
- `CLICKSTREAM_ADMIN.GENERATION_LOG` — batch execution log
- `CLICKSTREAM_ADMIN.GENERATE_INCREMENTAL_BATCH` — stored procedure
- `CLICKSTREAM_ADMIN.INCREMENTAL_DATA_TASK` — scheduled task (created SUSPENDED)

To clean up:
```sql
ALTER TASK FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.INCREMENTAL_DATA_TASK SUSPEND;
DROP TASK IF EXISTS FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.INCREMENTAL_DATA_TASK;
DROP PROCEDURE IF EXISTS FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.GENERATE_INCREMENTAL_BATCH();
DROP TABLE IF EXISTS FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.GENERATION_CONFIG;
DROP TABLE IF EXISTS FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.GENERATION_STATE;
DROP TABLE IF EXISTS FEATURE_STORE_DEMO_DEV.CLICKSTREAM_ADMIN.GENERATION_LOG;
```